# Recent Crossmatches Demo

Query the SCiMMA Rubin Crossmatch Service for the catalog crossmatches of
every object with an alert in a recent time window, grouped by object.

The endpoint is a read-only HTTP GET and is public/unauthenticated on DEV, so
no credentials are needed. See `docs/api/recent-crossmatch-api.md` for the full
parameter reference.

This notebook runs top to bottom against a live endpoint. Point `BASE_URL` at
the DEV host, or at a local `gunicorn`/`runserver` instance.

In [ ]:
import httpx
import pandas as pd

# DEV host. For a local server use e.g. 'http://127.0.0.1:8000'.
BASE_URL = 'https://crossmatch-dev.scimma.org'
ENDPOINT = f'{BASE_URL}/api/recent-crossmatches'


def get_recent(**params):
    """GET the endpoint with the given query params and return the parsed JSON."""
    resp = httpx.get(ENDPOINT, params=params, timeout=60.0)
    resp.raise_for_status()
    return resp.json()

## 1. Default window and `matches` detail

No parameters: the last 12 hours (by `ingest_time`), with each object's
position and its catalog matches (catalog, source id, separation).

In [ ]:
result = get_recent()
print('window:', result['window'])
print('time_field:', result['time_field'], '| detail:', result['detail'])
print('objects returned:', result['count'])

### Flatten the matches into a DataFrame

One row per (object, match): the object id and position alongside each match's
catalog, source id, and separation.

In [ ]:
rows = []
for obj in result['objects']:
    for match in obj.get('matches', []):
        rows.append({
            'diaObjectId': obj['diaObjectId'],
            'object_ra': obj.get('ra'),
            'object_dec': obj.get('dec'),
            'catalog_name': match['catalog_name'],
            'catalog_source_id': match['catalog_source_id'],
            'separation_arcsec': match['separation_arcsec'],
        })

matches_df = pd.DataFrame(rows)
matches_df

In [ ]:
# How many matches per catalog across all objects in the window?
if not matches_df.empty:
    display(matches_df['catalog_name'].value_counts())

## 2. An explicit window

Pass ISO-8601 UTC `start`/`end`. Here, the last 24 hours by observation time
(`event_time`) instead of ingest time.

In [ ]:
from datetime import datetime, timedelta, timezone

end = datetime.now(timezone.utc)
start = end - timedelta(hours=24)

windowed = get_recent(
    start=start.isoformat(),
    end=end.isoformat(),
    time_field='event_time',
)
print('objects in the last 24h (by event_time):', windowed['count'])

## 3. Full published payload

`detail=full` returns, for each match, exactly the payload the service
publishes over Hopskotch — including the nested `catalog_payload` of
catalog-specific columns. The `ra`/`dec` inside a match are the matched catalog
**source** coordinates, distinct from the object position.

In [ ]:
full = get_recent(detail='full', limit=100)
print('objects:', full['count'])

# Show the first full match payload, if any.
for obj in full['objects']:
    if obj.get('matches'):
        first_match = obj['matches'][0]
        break
else:
    first_match = None

first_match

In [ ]:
# Normalize the full payloads into a wide DataFrame: top-level match fields plus
# the flattened catalog_payload columns.
full_rows = []
for obj in full['objects']:
    for match in obj.get('matches', []):
        row = {k: v for k, v in match.items() if k != 'catalog_payload'}
        row.update(match.get('catalog_payload') or {})
        full_rows.append(row)

full_df = pd.DataFrame(full_rows)
full_df.head()